# Estudo técnico automatizado das tabelas DB2

Este notebook continua o padrão da `rotina-principal.ipynb` do Projeto.

## Arquitetura

1. cria a sessão pelo `GerenciadorSessaoSpark`;
2. em MODELAGEM carrega `desenv.env`;
3. cria a sessão com `db2=True`;
4. executa `gerenciador_sessao_spark_remoto.ipynb`;
5. cria `cliente_db2 = criar_cliente_db2_spark(env=dict(os.environ))`;
6. DB2 restringe os universos necessários;
7. Spark remoto faz perfil, cardinalidade, duplicidades e relacionamentos;
8. apenas um resultado pequeno retorna ao kernel local;
9. a única saída persistida é `estudo_tabelas_resultado.md`.

## Escopo

Somente:

- `DB2GFP.TRAN_RLZD_INST_PCT`
- `DB2GFP.INF_OPB_CT_CLI`
- `DB2GFP.CMPT_TRAN_RLZD_CC`
- `DB2GFP.CTGR_TRAN_OPB`
- `DB2GFP.GR_CTGR_TRAN`

Recorte inicial da tabela central:

**participante protegido + julho/2026 + BRL**

Não há regras financeiras, deduplicação analítica, principalidade, correção de categoria ou lógica do dashboard.

## Segurança

Nunca entram no relatório:

- `CD_CLI`;
- `CD_CLI_TITR_CT`;
- CPF/CNPJ;
- documento de contraparte;
- chave PIX;
- usuário/senha;
- identificadores brutos de conta.

Contas são apresentadas somente como `Conta 1`, `Conta 2`, ...

A linguagem do relatório diferencia:

- **METADADO TÉCNICO**
- **FATO OBSERVADO**
- **HIPÓTESE**
- **NÃO DETERMINADO**

In [ ]:
# ============================================================
# 1. Bootstrap local — mesmo padrão da rotina principal
# ============================================================
import os
import re
from pathlib import Path
from getpass import getpass
from traceback import format_exc

OUTPUT_MD = Path("estudo_tabelas_resultado.md")

spark = None
gerenciador_spark = None

def erro_local_sanitizado(exc):
    texto = f"{type(exc).__name__}: {str(exc)}"
    texto = re.sub(
        r"(?i)\b(password|senha|secret|token|keytab)\s*([=:])\s*([^\s,;]+)",
        r"\1\2***",
        texto,
    )
    return " ".join(texto.split())[:1000]

def gravar_execucao_incompleta(etapa, motivo, secoes=None):
    secoes = secoes or []
    linhas = [
        "# Estudo técnico das tabelas",
        "",
        "## EXECUÇÃO INCOMPLETA",
        "",
        f"**Etapa interrompida:** {etapa}",
        "",
        f"**Motivo técnico sanitizado:** {motivo}",
        "",
        "**Seções concluídas antes da falha:**",
    ]
    if secoes:
        linhas.extend(f"- {x}" for x in secoes)
    else:
        linhas.append("- Nenhuma.")
    OUTPUT_MD.write_text("\n".join(linhas) + "\n", encoding="utf-8")

try:
    from src.utils.gerenciador_sessao_spark_local import (
        GerenciadorSessaoSpark,
        ler_variavel_ambiente_local,
    )

    ambiente = ler_variavel_ambiente_local("AMBIENTE").upper()

    # Mantém o mesmo comportamento da rotina principal.
    if ambiente != "MODELAGEM":
        ambiente = "PRODUCAO"

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="estudo_tabelas",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "AMBIENTE": ambiente,
        },
        # Obrigatório em MODELAGEM para disponibilizar também
        # DB2_USER / DB2_PASSWORD / DB2_HOST / DB2_DATABASE ao Spark remoto.
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="12g",
        driver_cores=4,
        executor_memory="12g",
        executor_cores=4,
        num_executors=8,
        jars=[
            "/dados/shared/bin/ojdbc8.jar",
        ],
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "4g",
            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "240",
            "spark.sql.autoBroadcastJoinThreshold": "-1",
            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    CLIENTE_PROTEGIDO = os.environ.get("ESTUDO_CD_CLI")
    if not CLIENTE_PROTEGIDO:
        CLIENTE_PROTEGIDO = getpass(
            "Informe o participante protegido: "
        ).strip()

    if not CLIENTE_PROTEGIDO:
        raise ValueError("Participante protegido não informado.")

    PARAMETROS_ESTUDO = {
        "data_inicio": "2026-07-01",
        "data_fim_exclusivo": "2026-08-01",
        "moeda": "BRL",
    }

    # Somente objetos pequenos atravessam local -> Spark.
    spark.send_to_spark(CLIENTE_PROTEGIDO)
    spark.send_to_spark(PARAMETROS_ESTUDO)

    print("Bootstrap concluído: sessão Spark remota criada com suporte DB2.")
except Exception as exc:
    gravar_execucao_incompleta(
        "Bootstrap local",
        erro_local_sanitizado(exc),
    )
    raise

In [ ]:
# ============================================================
# 2. Carrega no remoto os clientes já implementados no Projeto
# ============================================================
try:
    if spark is None:
        raise RuntimeError("A sessão Spark não foi criada.")

    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb

except Exception as exc:
    gravar_execucao_incompleta(
        "Carga do gerenciador Spark remoto",
        erro_local_sanitizado(exc),
    )
    raise

In [ ]:
%%spark

# ============================================================
# 3. Contexto remoto e cliente DB2 oficial do Projeto
# ============================================================
import os
import re
from functools import reduce
from operator import or_

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StringType,
    DateType,
    TimestampType,
    ByteType,
    ShortType,
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
)
from pyspark.storagelevel import StorageLevel

env_spark = dict(os.environ)

# Esta criação exige DB2_USER, DB2_PASSWORD, DB2_HOST e DB2_DATABASE.
cliente_db2 = criar_cliente_db2_spark(env=env_spark)

T_MOV = "DB2GFP.TRAN_RLZD_INST_PCT"
T_CONTAS = "DB2GFP.INF_OPB_CT_CLI"
T_CMPT = "DB2GFP.CMPT_TRAN_RLZD_CC"
T_CAT = "DB2GFP.CTGR_TRAN_OPB"
T_GRUPO = "DB2GFP.GR_CTGR_TRAN"
TABELAS_ESTUDO = [T_MOV, T_CONTAS, T_CMPT, T_CAT, T_GRUPO]

RESULTADO_ESTUDO = {
    "execucao_ok": False,
    "etapa_interrompida": None,
    "erro": None,
    "secoes_concluidas": [],
    "identificacao": {
        "periodo": "julho/2026",
        "moeda": PARAMETROS_ESTUDO["moeda"],
        "tabelas": TABELAS_ESTUDO,
        "tecnologia": "GerenciadorSessaoSpark + BBMagic + ClientDb2Spark",
        "metadado_fisico": "METADADO_NAO_VALIDADO",
    },
    "resumo": {},
    "fontes": {},
    "instituicoes": {},
    "categorias": {},
    "agua": {},
    "relacionamentos": [],
    "anomalias": [],
    "questoes_abertas": [],
}

print("Contexto remoto concluído: ClientDb2Spark disponível.")

In [ ]:
%%spark

# ============================================================
# 4. Funções do estudo — executam somente no Spark remoto
# ============================================================

NUMERIC_TYPES = (
    ByteType,
    ShortType,
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
)

SENSIVEIS_RE = [
    re.compile(x, re.I)
    for x in [
        r"^CD_CLI$",
        r"^CD_CLI_TITR_CT$",
        r"CPF",
        r"CNPJ",
        r"DOCUMENT",
        r"(^|_)DOC($|_)",
        r"CHAVE.*PIX",
        r"PIX.*CHAVE",
        r"USUAR",
        r"LOGIN",
        r"SENHA",
        r"PASSWORD",
        r"(IDFR|IDENT).*(CTPT|CNTP|CONTRAP)",
        r"(CTPT|CNTP|CONTRAP).*(IDFR|IDENT|DOC)",
    ]
]

CONTA_RE = [
    re.compile(r"^NR_SEQL_CT_CLI$", re.I),
    re.compile(r"^CD_IDFR_CT$", re.I),
    re.compile(r"AG[EÊ]N?C", re.I),
    re.compile(r"^NR_.*(?:_CT(?:_|$)|CONTA)", re.I),
    re.compile(r"^CD_IDFR_.*(?:_CT(?:_|$)|CONTA)", re.I),
]

def etapa(nome):
    RESULTADO_ESTUDO["etapa_interrompida"] = nome
    print(f"[ETAPA] {nome}")

def concluida(nome):
    RESULTADO_ESTUDO["secoes_concluidas"].append(nome)

def adicionar_anomalia(classe, texto):
    RESULTADO_ESTUDO["anomalias"].append(
        {"classe": classe, "texto": str(texto)}
    )

def adicionar_questao(texto):
    texto = str(texto)
    if texto not in RESULTADO_ESTUDO["questoes_abertas"]:
        RESULTADO_ESTUDO["questoes_abertas"].append(texto)

def eh_sensivel(nome):
    return any(p.search(str(nome)) for p in SENSIVEIS_RE)

def eh_conta(nome):
    return any(p.search(str(nome)) for p in CONTA_RE)

def coluna_relatorio_segura(nome):
    return not eh_sensivel(nome) and not eh_conta(nome)

def valor_py(v):
    if v is None:
        return None
    if isinstance(v, (str, int, float, bool)):
        return v
    if hasattr(v, "isoformat"):
        try:
            return v.isoformat()
        except Exception:
            pass
    return str(v)

def rows_dict(df, limite=20):
    return [
        {
            k: valor_py(v)
            for k, v in r.asDict(recursive=True).items()
        }
        for r in df.limit(int(limite)).collect()
    ]

def score_nome(nome, regras):
    n = str(nome).upper()
    score = 0
    for peso, termos in regras:
        for termo in termos:
            if termo in n:
                score += peso
    return score

def probe_tabela(tabela):
    # Leitura de uma única linha: serve para obter schema/colunas sem varrer conteúdo.
    return cliente_db2.run_select(
        f"SELECT * FROM {tabela} FETCH FIRST 1 ROW ONLY",
        fetchsize=1,
        query_timeout=120,
    )

def sql_literal(valor, data_type=None):
    if valor is None:
        return "NULL"

    if isinstance(
        data_type,
        (ByteType, ShortType, IntegerType, LongType, FloatType, DoubleType, DecimalType),
    ):
        texto = str(valor).strip()
        if re.fullmatch(r"[-+]?\d+(?:\.\d+)?", texto):
            return texto

    texto = str(valor)
    return "'" + texto.replace("'", "''") + "'"

def schema_type_map(df):
    return {f.name: f.dataType for f in df.schema.fields}

def candidatos_temporais(df):
    candidatos = [
        f.name
        for f in df.schema.fields
        if isinstance(f.dataType, (DateType, TimestampType))
        and not eh_sensivel(f.name)
    ]
    candidatos.sort(
        key=lambda c: (
            -score_nome(
                c,
                [
                    (30, ["TRAN"]),
                    (20, ["MOV"]),
                    (10, ["DT_"]),
                    (10, ["DATA"]),
                    (8, ["DH_"]),
                ],
            ),
            c,
        )
    )
    return candidatos

def candidatos_moeda(df):
    textos = [
        f.name
        for f in df.schema.fields
        if isinstance(f.dataType, StringType)
        and not eh_sensivel(f.name)
        and not eh_conta(f.name)
    ]
    textos.sort(
        key=lambda c: (
            -score_nome(
                c,
                [
                    (50, ["MOED"]),
                    (30, ["CURR"]),
                    (5, ["CD_"]),
                ],
            ),
            c,
        )
    )

    com_nome_forte = [
        c
        for c in textos
        if score_nome(c, [(1, ["MOED", "CURR"])]) > 0
    ]
    if com_nome_forte:
        return com_nome_forte[:20]

    # Fallback controlado para não construir SQL com centenas de campos.
    return textos[:40]

def descobrir_e_carregar_movimentacoes():
    probe = probe_tabela(T_MOV)
    if "CD_CLI" not in probe.columns:
        raise RuntimeError(
            "CD_CLI não foi observado em TRAN_RLZD_INST_PCT."
        )

    temporais = candidatos_temporais(probe)
    moedas = candidatos_moeda(probe)

    if not temporais:
        raise RuntimeError(
            "Nenhuma coluna Date/Timestamp foi observada para delimitar julho."
        )
    if not moedas:
        raise RuntimeError(
            "Nenhuma coluna textual candidata a moeda foi observada."
        )

    tipos = schema_type_map(probe)
    cli_lit = sql_literal(
        CLIENTE_PROTEGIDO,
        tipos.get("CD_CLI"),
    )

    inicio = PARAMETROS_ESTUDO["data_inicio"]
    fim = PARAMETROS_ESTUDO["data_fim_exclusivo"]
    moeda = PARAMETROS_ESTUDO["moeda"]

    cond_data = " OR ".join(
        (
            f"({c} >= DATE('{inicio}') "
            f"AND {c} < DATE('{fim}'))"
        )
        for c in temporais
    )

    cond_moeda = " OR ".join(
        (
            f"(UPPER(TRIM(CAST({c} AS VARCHAR(64)))) = "
            f"'{moeda}')"
        )
        for c in moedas
    )

    # ÚNICA carga de conteúdo da tabela central.
    # É um superset ainda restrito por participante + julho + BRL.
    sql = (
        f"SELECT * FROM {T_MOV} "
        f"WHERE CD_CLI = {cli_lit} "
        f"AND ({cond_data}) "
        f"AND ({cond_moeda})"
    )

    superset = (
        cliente_db2.run_select(
            sql,
            fetchsize=10000,
            query_timeout=600,
        )
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    qt_super = superset.count()
    if qt_super == 0:
        raise RuntimeError(
            "Nenhum registro foi encontrado para participante + julho + BRL."
        )

    # Descobre a combinação data/moeda dentro do único universo já carregado.
    metricas = []
    for dt in temporais:
        if dt not in superset.columns:
            continue
        for md in moedas:
            if md not in superset.columns:
                continue
            cond = (
                (F.col(dt) >= F.lit(inicio))
                & (F.col(dt) < F.lit(fim))
                & (
                    F.upper(F.trim(F.col(md).cast("string")))
                    == F.lit(moeda)
                )
            )
            metricas.append(
                (
                    dt,
                    md,
                    superset.where(cond).count(),
                    score_nome(
                        dt,
                        [
                            (30, ["TRAN"]),
                            (20, ["MOV"]),
                            (10, ["DT_"]),
                            (8, ["DH_"]),
                        ],
                    )
                    + score_nome(
                        md,
                        [(50, ["MOED"]), (30, ["CURR"])],
                    ),
                )
            )

    validas = [x for x in metricas if x[2] > 0]
    if not validas:
        raise RuntimeError(
            "O superset possui registros, mas nenhuma combinação "
            "data/moeda candidata reproduziu julho + BRL."
        )

    validas.sort(key=lambda x: (-x[2], -x[3], x[0], x[1]))
    col_data, col_moeda, _, _ = validas[0]

    empatadas = [
        x
        for x in validas
        if x[2] == validas[0][2]
        and x[3] == validas[0][3]
    ]
    if len(empatadas) > 1:
        adicionar_questao(
            "Mais de uma combinação de data/moeda apresentou "
            "a mesma cobertura e score. A combinação usada operacionalmente foi "
            f"{col_data} + {col_moeda}."
        )

    mov = (
        superset.where(
            (F.col(col_data) >= F.lit(inicio))
            & (F.col(col_data) < F.lit(fim))
            & (
                F.upper(F.trim(F.col(col_moeda).cast("string")))
                == F.lit(moeda)
            )
        )
    )

    for c in ["CD_CLI", "CD_CLI_TITR_CT"]:
        if c in mov.columns:
            mov = mov.drop(c)

    mov = mov.persist(StorageLevel.MEMORY_AND_DISK)
    qt_mov = mov.count()

    return superset, mov, qt_super, qt_mov, col_data, col_moeda

def escolher_coluna_valor(df):
    candidatos = []
    for f in df.schema.fields:
        if not isinstance(f.dataType, NUMERIC_TYPES):
            continue
        if eh_sensivel(f.name) or eh_conta(f.name):
            continue

        score = score_nome(
            f.name,
            [
                (50, ["VL_"]),
                (50, ["VLR"]),
                (50, ["VALOR"]),
                (10, ["TRAN"]),
            ],
        )

        if any(
            token in f.name.upper()
            for token in ["CD_", "ID_", "SEQ", "QTD", "QT_"]
        ):
            score -= 30

        if score > 0:
            candidatos.append((score, f.name))

    if not candidatos:
        adicionar_questao(
            "Não foi possível determinar uma coluna candidata a valor."
        )
        return None

    candidatos.sort(key=lambda x: (-x[0], x[1]))

    if (
        len(candidatos) > 1
        and candidatos[0][0] == candidatos[1][0]
    ):
        adicionar_questao(
            "Mais de uma coluna numérica possui o mesmo score de valor: "
            f"{candidatos[0][1]} e {candidatos[1][1]}. "
            f"{candidatos[0][1]} foi usada apenas como hipótese operacional."
        )

    return candidatos[0][1]

def preenchimento_resumo(df, total):
    colunas = [
        c for c in df.columns if not eh_sensivel(c)
    ]
    if not colunas:
        return {
            "colunas_analisadas": 0,
            "colunas_100_vazias": [],
            "preenchimento_parcial": [],
        }

    row = df.agg(
        *[
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in colunas
        ]
    ).first().asDict()

    vazias = []
    parciais = []

    for c, qtd_nulos in row.items():
        qtd_nulos = int(qtd_nulos or 0)
        if qtd_nulos == total:
            vazias.append(c)
        elif qtd_nulos > 0:
            parciais.append(
                {
                    "coluna": c,
                    "nulos": qtd_nulos,
                    "pct_nulos": round(
                        qtd_nulos / total * 100.0,
                        2,
                    )
                    if total
                    else 0.0,
                }
            )

    parciais.sort(
        key=lambda x: (-x["pct_nulos"], x["coluna"])
    )

    return {
        "colunas_analisadas": len(colunas),
        "colunas_100_vazias": sorted(vazias),
        "preenchimento_parcial": parciais[:20],
    }

def perfil_cardinalidade(df, total):
    saida = []

    for c in df.columns:
        if not coluna_relatorio_segura(c):
            continue

        # Ao encontrar 101 valores, não calcula cardinalidade alta exata.
        qtd_ate_101 = (
            df.select(c)
            .where(F.col(c).isNotNull())
            .dropDuplicates([c])
            .limit(101)
            .count()
        )

        if qtd_ate_101 >= 101:
            saida.append(
                {
                    "coluna": c,
                    "cardinalidade": ">100",
                    "classe": "CARDINALIDADE > 100",
                    "valores": [],
                }
            )
            continue

        frequencias = (
            df.groupBy(c)
            .agg(F.count("*").alias("quantidade"))
            .withColumn(
                "percentual",
                F.round(
                    F.col("quantidade")
                    / F.lit(total)
                    * F.lit(100.0),
                    2,
                ),
            )
            .orderBy(F.desc("quantidade"))
        )

        limite = 30 if qtd_ate_101 <= 30 else 20

        saida.append(
            {
                "coluna": c,
                "cardinalidade": int(qtd_ate_101),
                "classe": (
                    "CARDINALIDADE <= 30"
                    if qtd_ate_101 <= 30
                    else "CARDINALIDADE 31-100"
                ),
                "valores": rows_dict(
                    frequencias,
                    limite,
                ),
            }
        )

    return saida

def testar_chave(df, chaves, total, nome):
    if not all(c in df.columns for c in chaves):
        return {
            "nome": nome,
            "chaves": chaves,
            "status": "NÃO DETERMINADO",
            "motivo": "Uma ou mais colunas não existem.",
        }

    g = (
        df.groupBy(*chaves)
        .agg(F.count("*").alias("QT"))
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    row = g.agg(
        F.count("*").alias("COMBINACOES"),
        F.sum(
            F.when(F.col("QT") > 1, 1).otherwise(0)
        ).alias("REPETICOES"),
        F.max("QT").alias("MAIOR"),
    ).first()

    repeticoes = int(row["REPETICOES"] or 0)

    out = {
        "nome": nome,
        "chaves": chaves,
        "status": "FATO OBSERVADO",
        "linhas": int(total),
        "combinacoes": int(row["COMBINACOES"] or 0),
        "repeticoes": repeticoes,
        "maior_multiplicidade": int(row["MAIOR"] or 0),
        "exemplos": [],
    }

    if repeticoes:
        rep = (
            g.where(F.col("QT") > 1)
            .orderBy(F.desc("QT"))
        )

        cols_ex = [
            c for c in chaves if not eh_conta(c)
        ]
        if cols_ex:
            out["exemplos"] = rows_dict(
                rep.select(*cols_ex, "QT"),
                10,
            )

        adicionar_anomalia(
            "FATO OBSERVADO",
            f"{nome}: {repeticoes} combinação(ões) repetida(s).",
        )

    g.unpersist()
    return out

def relacionamento(esq, dir_, chaves, nome):
    if not all(
        c in esq.columns and c in dir_.columns
        for c in chaves
    ):
        return {
            "nome": nome,
            "chaves": chaves,
            "status": "NÃO DETERMINADO",
            "motivo": "Uma ou mais colunas não existem nos dois lados.",
        }

    esq_null = esq.where(
        reduce(
            or_,
            [F.col(c).isNull() for c in chaves],
        )
    ).count()

    dir_null = dir_.where(
        reduce(
            or_,
            [F.col(c).isNull() for c in chaves],
        )
    ).count()

    esq_ok = esq.where(
        reduce(
            lambda a, b: a & b,
            [F.col(c).isNotNull() for c in chaves],
        )
    )
    dir_ok = dir_.where(
        reduce(
            lambda a, b: a & b,
            [F.col(c).isNotNull() for c in chaves],
        )
    )

    e = (
        esq_ok.groupBy(*chaves)
        .agg(F.count("*").alias("QT_ORIGEM"))
    )
    d = (
        dir_ok.groupBy(*chaves)
        .agg(F.count("*").alias("QT_DESTINO"))
    )

    r = (
        e.join(d, chaves, "full")
        .fillna(0, ["QT_ORIGEM", "QT_DESTINO"])
    )

    row = r.agg(
        F.count("*").alias("CHAVES_UNIAO"),
        F.sum(
            F.when(F.col("QT_ORIGEM") > 0, 1).otherwise(0)
        ).alias("CHAVES_ORIGEM"),
        F.sum(
            F.when(F.col("QT_DESTINO") > 0, 1).otherwise(0)
        ).alias("CHAVES_DESTINO"),
        F.sum(
            F.when(
                (F.col("QT_ORIGEM") > 0)
                & (F.col("QT_DESTINO") == 0),
                1,
            ).otherwise(0)
        ).alias("SEM"),
        F.sum(
            F.when(
                (F.col("QT_ORIGEM") == 1)
                & (F.col("QT_DESTINO") == 1),
                1,
            ).otherwise(0)
        ).alias("UM_UM"),
        F.sum(
            F.when(
                (F.col("QT_ORIGEM") == 1)
                & (F.col("QT_DESTINO") > 1),
                1,
            ).otherwise(0)
        ).alias("UM_N"),
        F.sum(
            F.when(
                (F.col("QT_ORIGEM") > 1)
                & (F.col("QT_DESTINO") == 1),
                1,
            ).otherwise(0)
        ).alias("N_UM"),
        F.sum(
            F.when(
                (F.col("QT_ORIGEM") > 1)
                & (F.col("QT_DESTINO") > 1),
                1,
            ).otherwise(0)
        ).alias("N_N"),
        F.sum(
            F.when(F.col("QT_ORIGEM") > 1, 1).otherwise(0)
        ).alias("REP_O"),
        F.sum(
            F.when(F.col("QT_DESTINO") > 1, 1).otherwise(0)
        ).alias("REP_D"),
    ).first()

    out = {
        "nome": nome,
        "chaves": chaves,
        "status": "FATO OBSERVADO",
        "chaves_origem": int(row["CHAVES_ORIGEM"] or 0),
        "chaves_destino": int(row["CHAVES_DESTINO"] or 0),
        "linhas_origem_com_chave_nula": int(esq_null),
        "linhas_destino_com_chave_nula": int(dir_null),
        "sem_match": int(row["SEM"] or 0),
        "um_um": int(row["UM_UM"] or 0),
        "um_n": int(row["UM_N"] or 0),
        "n_um": int(row["N_UM"] or 0),
        "n_n": int(row["N_N"] or 0),
        "chaves_repetidas_origem": int(row["REP_O"] or 0),
        "chaves_repetidas_destino": int(row["REP_D"] or 0),
        "exemplos": [],
    }

    if (
        out["sem_match"]
        or out["um_n"]
        or out["n_um"]
        or out["n_n"]
    ):
        ex = r.where(
            (F.col("QT_DESTINO") == 0)
            | (F.col("QT_ORIGEM") > 1)
            | (F.col("QT_DESTINO") > 1)
        )

        cols_ex = [
            c for c in chaves if not eh_conta(c)
        ]

        out["exemplos"] = rows_dict(
            ex.select(
                *cols_ex,
                "QT_ORIGEM",
                "QT_DESTINO",
            ),
            10,
        )

    return out

def leitura_dirigida_por_chaves(
    tabela,
    probe,
    chaves_df,
    chaves,
    tamanho_lote=250,
    max_chaves=10000,
):
    if not all(c in probe.columns for c in chaves):
        raise RuntimeError(
            f"{tabela}: chave dirigida ausente: {chaves}"
        )

    valores = (
        chaves_df.select(*chaves)
        .where(
            reduce(
                lambda a, b: a & b,
                [F.col(c).isNotNull() for c in chaves],
            )
        )
        .dropDuplicates()
    )

    qtd = valores.limit(max_chaves + 1).count()

    if qtd == 0:
        return cliente_db2.run_select(
            f"SELECT * FROM {tabela} WHERE 1 = 0",
            fetchsize=1,
            query_timeout=120,
        ), "SEM_CHAVES"

    if qtd > max_chaves:
        raise RuntimeError(
            f"{tabela}: mais de {max_chaves} chaves seriam necessárias "
            "para leitura dirigida; varredura ampla foi bloqueada."
        )

    rows = valores.collect()
    tipos = schema_type_map(probe)

    dataframes = []

    for inicio in range(0, len(rows), tamanho_lote):
        lote = rows[inicio:inicio + tamanho_lote]
        predicados = []

        for r in lote:
            partes = [
                (
                    f"{c} = "
                    f"{sql_literal(r[c], tipos.get(c))}"
                )
                for c in chaves
            ]
            predicados.append(
                "(" + " AND ".join(partes) + ")"
            )

        sql = (
            f"SELECT * FROM {tabela} "
            f"WHERE "
            + " OR ".join(predicados)
        )

        dataframes.append(
            cliente_db2.run_select(
                sql,
                fetchsize=10000,
                query_timeout=600,
            )
        )

    df = dataframes[0]
    for parte in dataframes[1:]:
        df = df.unionByName(
            parte,
            allowMissingColumns=True,
        )

    return df, "PREDICADOS_DIRIGIDOS_EM_LOTES"

def rotular_contas(mov, contas):
    candidatos = [
        c
        for c in ["NR_SEQL_CT_CLI", "CD_IDFR_CT"]
        if c in mov.columns and c in contas.columns
    ]

    if not candidatos:
        adicionar_questao(
            "Não foi possível formar rótulo comum de conta "
            "entre movimentações e INF_OPB_CT_CLI."
        )
        return mov, contas, None, []

    mapa = (
        mov.select(*candidatos)
        .unionByName(
            contas.select(*candidatos)
        )
        .dropDuplicates()
        .withColumn(
            "CONTA_PROTEGIDA",
            F.concat(
                F.lit("Conta "),
                F.row_number()
                .over(
                    Window.orderBy(
                        *[
                            F.col(c).asc_nulls_last()
                            for c in candidatos
                        ]
                    )
                )
                .cast("string"),
            ),
        )
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    return (
        mov.join(F.broadcast(mapa), candidatos, "left"),
        contas.join(F.broadcast(mapa), candidatos, "left"),
        mapa,
        candidatos,
    )

def colunas_textuais_seguras(df):
    return [
        f.name
        for f in df.schema.fields
        if isinstance(f.dataType, StringType)
        and coluna_relatorio_segura(f.name)
    ]

def texto_normalizado(coluna):
    return F.upper(
        F.translate(
            F.coalesce(
                F.col(coluna).cast("string"),
                F.lit(""),
            ),
            "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ",
            "AAAAAEEEEIIIIOOOOOUUUUC",
        )
    )

def escolher_descricao_agua(categorias):
    encontrados = []

    for c in colunas_textuais_seguras(categorias):
        if (
            categorias.where(
                texto_normalizado(c).contains("AGUA")
            )
            .limit(1)
            .count()
        ):
            encontrados.append(c)

    if not encontrados:
        return None

    encontrados.sort(
        key=lambda c: (
            -score_nome(
                c,
                [
                    (50, ["DS_", "DESC"]),
                    (20, ["CTGR"]),
                    (10, ["NM_"]),
                ],
            ),
            c,
        )
    )

    if len(encontrados) > 1:
        adicionar_questao(
            "O texto Água aparece em mais de uma coluna textual "
            "da dimensão de categoria: "
            + ", ".join(encontrados)
            + f". {encontrados[0]} foi usada como hipótese operacional."
        )

    return encontrados[0]

def duplicidade_exata(df, colunas):
    if not colunas:
        return {
            "grupos": 0,
            "linhas": 0,
            "exemplos": [],
        }

    g = (
        df.groupBy(*colunas)
        .agg(F.count("*").alias("QT"))
        .where(F.col("QT") > 1)
    )

    row = g.agg(
        F.count("*").alias("GRUPOS"),
        F.sum("QT").alias("LINHAS"),
    ).first()

    grupos = int(row["GRUPOS"] or 0)
    linhas = int(row["LINHAS"] or 0)

    cols_ex = [
        c for c in colunas if not eh_conta(c)
    ]

    exemplos = (
        rows_dict(
            g.select(*cols_ex, "QT")
            .orderBy(F.desc("QT")),
            10,
        )
        if grupos
        else []
    )

    return {
        "grupos": grupos,
        "linhas": linhas,
        "exemplos": exemplos,
    }

def score_relacionamento(rel):
    if rel.get("status") != "FATO OBSERVADO":
        return -10**18

    # Quanto maior, melhor para uso operacional em auditoria.
    return (
        rel.get("um_um", 0) * 100
        - rel.get("sem_match", 0) * 50
        - rel.get("um_n", 0) * 20
        - rel.get("n_um", 0) * 10
        - rel.get("n_n", 0) * 30
    )

print("Funções do estudo carregadas.")

In [ ]:
%%spark

# ============================================================
# 5. Execução completa do estudo
# ============================================================
try:
    # --------------------------------------------------------
    etapa("TRAN_RLZD_INST_PCT — carga única e perfil")
    # --------------------------------------------------------
    (
        mov_superset,
        mov,
        QT_SUPERSET,
        QT_MOV,
        COL_DATA,
        COL_MOEDA,
    ) = descobrir_e_carregar_movimentacoes()

    COL_VALOR = escolher_coluna_valor(mov)

    f_mov = {
        "volume": QT_MOV,
        "volume_superset": QT_SUPERSET,
        "grao": (
            "HIPÓTESE: uma linha representa uma ocorrência de movimentação; "
            "as hipóteses de identidade são testadas empiricamente."
        ),
        "mapeamento_operacional": {
            "data": COL_DATA,
            "moeda": COL_MOEDA,
            "valor": COL_VALOR or "NÃO DETERMINADO",
        },
        "preenchimento": preenchimento_resumo(
            mov,
            QT_MOV,
        ),
        "dominios": perfil_cardinalidade(
            mov,
            QT_MOV,
        ),
        "chaves": [],
    }

    for chaves, nome in [
        (
            ["NR_TRAN_INST_PCT"],
            "NR_TRAN_INST_PCT",
        ),
        (
            ["NR_TRAN_INST_PCT", "NR_PTC"],
            "NR_TRAN_INST_PCT + NR_PTC",
        ),
    ]:
        f_mov["chaves"].append(
            testar_chave(
                mov,
                chaves,
                QT_MOV,
                nome,
            )
        )

    RESULTADO_ESTUDO["fontes"]["TRAN_RLZD_INST_PCT"] = f_mov
    RESULTADO_ESTUDO["resumo"]["movimentacoes"] = QT_MOV
    concluida("Fonte TRAN_RLZD_INST_PCT")

    # --------------------------------------------------------
    etapa("INF_OPB_CT_CLI — contas e instituições")
    # --------------------------------------------------------
    probe_contas = probe_tabela(T_CONTAS)

    if "CD_CLI" in probe_contas.columns:
        COL_CLIENTE_CONTAS = "CD_CLI"
    elif "CD_CLI_TITR_CT" in probe_contas.columns:
        COL_CLIENTE_CONTAS = "CD_CLI_TITR_CT"
    else:
        raise RuntimeError(
            "INF_OPB_CT_CLI não possui coluna de participante esperada."
        )

    tipos_contas = schema_type_map(probe_contas)
    cli_lit_contas = sql_literal(
        CLIENTE_PROTEGIDO,
        tipos_contas.get(COL_CLIENTE_CONTAS),
    )

    # Única leitura de conteúdo da tabela de contas para o participante.
    contas = (
        cliente_db2.run_select(
            (
                f"SELECT * FROM {T_CONTAS} "
                f"WHERE {COL_CLIENTE_CONTAS} = {cli_lit_contas}"
            ),
            fetchsize=10000,
            query_timeout=600,
        )
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    for c in ["CD_CLI", "CD_CLI_TITR_CT"]:
        if c in contas.columns:
            contas = contas.drop(c)

    QT_CONTAS = contas.count()

    mov_ant = mov
    contas_ant = contas

    (
        mov,
        contas,
        mapa_conta,
        COLS_CONTA,
    ) = rotular_contas(
        mov,
        contas,
    )

    if mapa_conta is not None:
        mov = mov.persist(StorageLevel.MEMORY_AND_DISK)
        contas = contas.persist(StorageLevel.MEMORY_AND_DISK)
        mov.count()
        contas.count()

        # Mantém o superset, mas libera versões intermediárias substituídas.
        mov_ant.unpersist()
        contas_ant.unpersist()

    f_contas = {
        "volume": QT_CONTAS,
        "grao": (
            "HIPÓTESE: registro de vínculo/estado de conta; "
            "uma linha não é presumida como uma conta única."
        ),
        "preenchimento": preenchimento_resumo(
            contas,
            QT_CONTAS,
        ),
        "dominios": perfil_cardinalidade(
            contas,
            QT_CONTAS,
        ),
        "chaves": [],
        "instituicoes": [],
        "marcas": [],
    }

    for chaves, nome in [
        (
            ["NR_SEQL_CT_CLI"],
            "NR_SEQL_CT_CLI",
        ),
        (
            ["CD_IDFR_CT"],
            "CD_IDFR_CT",
        ),
        (
            ["NR_SEQL_CT_CLI", "CD_IDFR_CT"],
            "NR_SEQL_CT_CLI + CD_IDFR_CT",
        ),
        (
            [
                "CD_INST_PCT",
                "NR_SEQL_CT_CLI",
                "CD_IDFR_CT",
            ],
            "instituição + conta",
        ),
        (
            [
                "CD_INST_PCT",
                "NR_MCA_PCT_OPB",
                "NR_SEQL_CT_CLI",
                "CD_IDFR_CT",
            ],
            "instituição + marca + conta",
        ),
    ]:
        f_contas["chaves"].append(
            testar_chave(
                contas,
                chaves,
                QT_CONTAS,
                nome,
            )
        )

    if "CD_INST_PCT" in contas.columns:
        inst = (
            contas.groupBy("CD_INST_PCT")
            .agg(
                F.count("*").alias("registros")
            )
        )

        if "NR_MCA_PCT_OPB" in contas.columns:
            marcas_inst = (
                contas.select(
                    "CD_INST_PCT",
                    "NR_MCA_PCT_OPB",
                )
                .dropDuplicates()
                .groupBy("CD_INST_PCT")
                .agg(
                    F.count("*").alias("marcas")
                )
            )
            inst = inst.join(
                marcas_inst,
                "CD_INST_PCT",
                "left",
            )
        else:
            inst = inst.withColumn(
                "marcas",
                F.lit(None).cast("long"),
            )

        if "CONTA_PROTEGIDA" in contas.columns:
            contas_inst = (
                contas.select(
                    "CD_INST_PCT",
                    "CONTA_PROTEGIDA",
                )
                .dropDuplicates()
                .groupBy("CD_INST_PCT")
                .agg(
                    F.count("*").alias(
                        "contas_candidatas"
                    )
                )
            )
            inst = inst.join(
                contas_inst,
                "CD_INST_PCT",
                "left",
            )
        else:
            inst = inst.withColumn(
                "contas_candidatas",
                F.lit(None).cast("long"),
            )

        f_contas["instituicoes"] = rows_dict(
            inst.orderBy("CD_INST_PCT"),
            200,
        )

    if "NR_MCA_PCT_OPB" in contas.columns:
        f_contas["marcas"] = rows_dict(
            contas.groupBy("NR_MCA_PCT_OPB")
            .agg(
                F.count("*").alias("registros")
            )
            .orderBy(F.desc("registros")),
            200,
        )

    RESULTADO_ESTUDO["fontes"]["INF_OPB_CT_CLI"] = f_contas

    RESULTADO_ESTUDO["resumo"]["instituicoes_estrutura"] = (
        contas.select("CD_INST_PCT")
        .dropDuplicates()
        .count()
        if "CD_INST_PCT" in contas.columns
        else None
    )

    RESULTADO_ESTUDO["resumo"]["marcas_estrutura"] = (
        contas.select("NR_MCA_PCT_OPB")
        .dropDuplicates()
        .count()
        if "NR_MCA_PCT_OPB" in contas.columns
        else None
    )

    # Estrutura de contas x instituições movimentadas.
    if (
        "CD_INST_PCT" in contas.columns
        and "CD_INST_PCT" in mov.columns
    ):
        inst_estrutura = (
            contas.select("CD_INST_PCT")
            .dropDuplicates()
            .withColumn(
                "na_estrutura_contas",
                F.lit("SIM"),
            )
        )

        inst_julho = (
            mov.select("CD_INST_PCT")
            .dropDuplicates()
            .withColumn(
                "nas_movimentacoes_julho",
                F.lit("SIM"),
            )
        )

        comparacao = (
            inst_estrutura.join(
                inst_julho,
                "CD_INST_PCT",
                "full",
            )
            .fillna(
                "NÃO",
                [
                    "na_estrutura_contas",
                    "nas_movimentacoes_julho",
                ],
            )
            .orderBy("CD_INST_PCT")
        )

        RESULTADO_ESTUDO["instituicoes"]["comparacao"] = (
            rows_dict(comparacao, 200)
        )

        RESULTADO_ESTUDO["resumo"]["instituicoes_movimentadas"] = (
            inst_julho.count()
        )

        RESULTADO_ESTUDO["instituicoes"]["estrutura_sem_movimento"] = (
            comparacao.where(
                (F.col("na_estrutura_contas") == "SIM")
                & (
                    F.col("nas_movimentacoes_julho")
                    == "NÃO"
                )
            ).count()
        )
    else:
        adicionar_questao(
            "Não foi possível comparar instituições da estrutura "
            "de contas com as movimentações."
        )

    if (
        "CD_INST_PCT" in contas.columns
        and "NR_MCA_PCT_OPB" in contas.columns
    ):
        marcas_por_inst = (
            contas.select(
                "CD_INST_PCT",
                "NR_MCA_PCT_OPB",
            )
            .dropDuplicates()
            .groupBy("CD_INST_PCT")
            .agg(F.count("*").alias("qt_marcas"))
        )

        multi = marcas_por_inst.where(
            F.col("qt_marcas") > 1
        )

        if multi.limit(1).count():
            RESULTADO_ESTUDO["instituicoes"]["multimarcas"] = (
                rows_dict(
                    multi.orderBy(F.desc("qt_marcas")),
                    20,
                )
            )

            adicionar_anomalia(
                "FATO OBSERVADO",
                f"{multi.count()} instituição(ões) aparece(m) "
                "associada(s) a mais de uma marca.",
            )

        adicionar_questao(
            "Instituição e marca representam entidades diferentes? "
            "A associação foi observada, mas a semântica não é determinada."
        )

    concluida("Fonte INF_OPB_CT_CLI e instituições")

    # --------------------------------------------------------
    etapa("Relacionamentos movimentação x conta")
    # --------------------------------------------------------
    for chaves in [
        ["CD_INST_PCT"],
        ["CD_INST_PCT", "NR_MCA_PCT_OPB"],
        ["CD_INST_PCT", "NR_SEQL_CT_CLI"],
        ["CD_INST_PCT", "CD_IDFR_CT"],
        [
            "CD_INST_PCT",
            "NR_SEQL_CT_CLI",
            "CD_IDFR_CT",
        ],
    ]:
        RESULTADO_ESTUDO["relacionamentos"].append(
            relacionamento(
                mov,
                contas,
                chaves,
                "TRAN_RLZD_INST_PCT → INF_OPB_CT_CLI",
            )
        )

    concluida("Relacionamentos movimentação x conta")

    # --------------------------------------------------------
    etapa("CMPT_TRAN_RLZD_CC — leitura dirigida única")
    # --------------------------------------------------------
    probe_cmpt = probe_tabela(T_CMPT)

    if (
        "NR_TRAN_INST_PCT" in mov.columns
        and "NR_TRAN_INST_PCT" in probe_cmpt.columns
    ):
        cmpt, estrategia_cmpt = leitura_dirigida_por_chaves(
            T_CMPT,
            probe_cmpt,
            mov.select("NR_TRAN_INST_PCT"),
            ["NR_TRAN_INST_PCT"],
            tamanho_lote=250,
            max_chaves=10000,
        )

        cmpt = cmpt.persist(
            StorageLevel.MEMORY_AND_DISK
        )
        QT_CMPT = cmpt.count()
    else:
        cmpt = None
        QT_CMPT = None
        estrategia_cmpt = "NÃO DETERMINADO"

    rel_cmpt = []

    if cmpt is not None:
        for chaves in [
            ["NR_TRAN_INST_PCT"],
            ["NR_TRAN_INST_PCT", "NR_PTC"],
        ]:
            rel = relacionamento(
                mov,
                cmpt,
                chaves,
                "TRAN_RLZD_INST_PCT → CMPT_TRAN_RLZD_CC",
            )
            rel["estrategia_leitura"] = estrategia_cmpt
            rel_cmpt.append(rel)
            RESULTADO_ESTUDO["relacionamentos"].append(
                rel
            )

        RESULTADO_ESTUDO["fontes"]["CMPT_TRAN_RLZD_CC"] = {
            "volume": QT_CMPT,
            "grao": (
                "HIPÓTESE: linha de complemento associada a uma movimentação; "
                "0/1/N é medido nas chaves candidatas."
            ),
            "estrategia_leitura": estrategia_cmpt,
            "preenchimento": preenchimento_resumo(
                cmpt,
                QT_CMPT,
            ),
            "dominios": perfil_cardinalidade(
                cmpt,
                QT_CMPT,
            ),
            "relacionamentos": rel_cmpt,
        }
    else:
        RESULTADO_ESTUDO["fontes"]["CMPT_TRAN_RLZD_CC"] = {
            "volume": None,
            "grao": "NÃO DETERMINADO",
            "estrategia_leitura": estrategia_cmpt,
            "relacionamentos": [],
        }

        adicionar_questao(
            "Não foi possível carregar complementos de forma dirigida."
        )

    concluida("Fonte CMPT_TRAN_RLZD_CC")

    # --------------------------------------------------------
    etapa("CTGR_TRAN_OPB — somente códigos usados")
    # --------------------------------------------------------
    probe_cat = probe_tabela(T_CAT)

    if not all(
        c in mov.columns
        for c in [
            "CD_CTGR_TRAN",
            "CD_CTGR_TRAN_OGNL",
        ]
    ):
        raise RuntimeError(
            "As colunas de categoria original/vigente "
            "não foram observadas na movimentação."
        )

    cod_cat = (
        mov.select(
            F.col("CD_CTGR_TRAN").alias(
                "CD_CTGR_TRAN"
            )
        )
        .unionByName(
            mov.select(
                F.col("CD_CTGR_TRAN_OGNL").alias(
                    "CD_CTGR_TRAN"
                )
            )
        )
        .where(
            F.col("CD_CTGR_TRAN").isNotNull()
        )
        .dropDuplicates()
    )

    categorias, estrategia_cat = leitura_dirigida_por_chaves(
        T_CAT,
        probe_cat,
        cod_cat,
        ["CD_CTGR_TRAN"],
        tamanho_lote=250,
        max_chaves=5000,
    )

    categorias = categorias.persist(
        StorageLevel.MEMORY_AND_DISK
    )
    QT_CAT = categorias.count()

    f_cat = {
        "volume": QT_CAT,
        "grao": (
            "HIPÓTESE: registro de domínio de categoria; "
            "a unicidade do código é testada empiricamente."
        ),
        "estrategia_leitura": estrategia_cat,
        "preenchimento": preenchimento_resumo(
            categorias,
            QT_CAT,
        ),
        "dominios": perfil_cardinalidade(
            categorias,
            QT_CAT,
        ),
        "chaves": [
            testar_chave(
                categorias,
                ["CD_CTGR_TRAN"],
                QT_CAT,
                "CD_CTGR_TRAN",
            )
        ],
        "variacoes_por_codigo": [],
    }

    for c in categorias.columns:
        if (
            c == "CD_CTGR_TRAN"
            or not coluna_relatorio_segura(c)
        ):
            continue

        pares = (
            categorias.select(
                "CD_CTGR_TRAN",
                c,
            )
            .dropDuplicates()
        )

        por_codigo = (
            pares.groupBy("CD_CTGR_TRAN")
            .agg(
                F.count("*").alias("QT_VALORES")
            )
        )

        row = por_codigo.agg(
            F.max("QT_VALORES").alias("MAIOR"),
            F.sum(
                F.when(
                    F.col("QT_VALORES") > 1,
                    1,
                ).otherwise(0)
            ).alias("COM_VARIACAO"),
        ).first()

        item = {
            "coluna": c,
            "maior_qt_valores_por_codigo": int(
                row["MAIOR"] or 0
            ),
            "codigos_com_variacao": int(
                row["COM_VARIACAO"] or 0
            ),
            "exemplos": [],
        }

        if item["codigos_com_variacao"]:
            item["exemplos"] = rows_dict(
                por_codigo.where(
                    F.col("QT_VALORES") > 1
                )
                .orderBy(F.desc("QT_VALORES")),
                10,
            )

            adicionar_anomalia(
                "FATO OBSERVADO",
                f"CD_CTGR_TRAN possui mais de um valor de {c} "
                f"para {item['codigos_com_variacao']} código(s).",
            )

        f_cat["variacoes_por_codigo"].append(
            item
        )

    COL_DESC_CAT = escolher_descricao_agua(
        categorias
    )

    f_cat["descricao_detectada"] = (
        COL_DESC_CAT or "NÃO DETERMINADO"
    )

    RESULTADO_ESTUDO["fontes"]["CTGR_TRAN_OPB"] = (
        f_cat
    )
    RESULTADO_ESTUDO["categorias"]["descricao_coluna"] = (
        COL_DESC_CAT
    )

    if not COL_DESC_CAT:
        adicionar_questao(
            "Não foi possível determinar automaticamente "
            "a coluna de descrição da categoria pelo caso Água."
        )

    concluida("Fonte CTGR_TRAN_OPB")

    # --------------------------------------------------------
    etapa("GR_CTGR_TRAN — somente grupos necessários")
    # --------------------------------------------------------
    probe_grupo = probe_tabela(T_GRUPO)

    comuns_grupo = [
        c
        for c in categorias.columns
        if c in probe_grupo.columns
        and c != "CD_CTGR_TRAN"
        and coluna_relatorio_segura(c)
    ]

    # Não escolhe a relação apenas pelo nome.
    # Testa empiricamente candidatos plausíveis e seleciona o melhor comportamento observado.
    candidatos_grupo = sorted(
        comuns_grupo,
        key=lambda c: (
            -score_nome(
                c,
                [
                    (50, ["GR_CTGR"]),
                    (30, ["GRUPO"]),
                    (20, ["GR_"]),
                    (10, ["CTGR"]),
                ],
            ),
            c,
        ),
    )[:8]

    testes_grupo = []
    grupos_escolhido = None
    COL_GRUPO = None
    melhor_score_grupo = None

    for candidato in candidatos_grupo:
        valores_grupo = (
            categorias.select(candidato)
            .where(F.col(candidato).isNotNull())
            .dropDuplicates()
        )

        try:
            grupos_cand, estrategia_cand = leitura_dirigida_por_chaves(
                T_GRUPO,
                probe_grupo,
                valores_grupo,
                [candidato],
                tamanho_lote=250,
                max_chaves=5000,
            )

            grupos_cand = grupos_cand.persist(
                StorageLevel.MEMORY_AND_DISK
            )
            qt_grupo_cand = grupos_cand.count()

            rel_cand = relacionamento(
                categorias,
                grupos_cand,
                [candidato],
                "CTGR_TRAN_OPB → GR_CTGR_TRAN",
            )
            rel_cand["estrategia_leitura"] = estrategia_cand
            rel_cand["volume_destino"] = qt_grupo_cand

            score_cand = score_relacionamento(rel_cand)

            testes_grupo.append({
                "coluna": candidato,
                "score_observado": score_cand,
                "relacionamento": rel_cand,
            })

            RESULTADO_ESTUDO["relacionamentos"].append(rel_cand)

            if (
                melhor_score_grupo is None
                or score_cand > melhor_score_grupo
            ):
                if grupos_escolhido is not None:
                    try:
                        grupos_escolhido.unpersist()
                    except Exception:
                        pass

                melhor_score_grupo = score_cand
                COL_GRUPO = candidato
                grupos_escolhido = grupos_cand
            else:
                grupos_cand.unpersist()

        except Exception as exc_cand:
            testes_grupo.append({
                "coluna": candidato,
                "score_observado": None,
                "status": "NÃO DETERMINADO",
                "motivo": f"{type(exc_cand).__name__}: {str(exc_cand)}"[:300],
            })

    if grupos_escolhido is not None:
        grupos = grupos_escolhido
        QT_GRUPO = grupos.count()

        # Se houver empate, a escolha continua sendo hipótese operacional.
        scores_validos = sorted(
            [
                x["score_observado"]
                for x in testes_grupo
                if x.get("score_observado") is not None
            ],
            reverse=True,
        )

        if (
            len(scores_validos) > 1
            and scores_validos[0] == scores_validos[1]
        ):
            adicionar_questao(
                "Mais de uma coluna candidata apresentou o mesmo comportamento "
                "para categoria → grupo. "
                f"{COL_GRUPO} foi mantida apenas como hipótese operacional."
            )

        RESULTADO_ESTUDO["fontes"]["GR_CTGR_TRAN"] = {
            "volume": QT_GRUPO,
            "grao": (
                "HIPÓTESE: registro de domínio de grupo."
            ),
            "chave_relacionamento_candidata": COL_GRUPO,
            "preenchimento": preenchimento_resumo(
                grupos,
                QT_GRUPO,
            ),
            "dominios": perfil_cardinalidade(
                grupos,
                QT_GRUPO,
            ),
            "testes_relacionamento": testes_grupo,
        }

        # Texto de grupo repetido em códigos diferentes.
        for c in colunas_textuais_seguras(grupos):
            if c == COL_GRUPO:
                continue

            rev = (
                grupos.select(
                    COL_GRUPO,
                    c,
                )
                .dropDuplicates()
                .groupBy(c)
                .agg(
                    F.count("*").alias(
                        "QT_CODIGOS_GRUPO"
                    )
                )
                .where(
                    F.col("QT_CODIGOS_GRUPO") > 1
                )
            )

            if rev.limit(1).count():
                adicionar_anomalia(
                    "FATO OBSERVADO",
                    f"GR_CTGR_TRAN: há textos de {c} "
                    "associados a mais de um código de grupo.",
                )
    else:
        RESULTADO_ESTUDO["fontes"]["GR_CTGR_TRAN"] = {
            "volume": None,
            "grao": "NÃO DETERMINADO",
            "chave_relacionamento_candidata": "NÃO DETERMINADO",
            "testes_relacionamento": testes_grupo,
        }

        adicionar_questao(
            "Nenhuma relação categoria → grupo pôde ser determinada "
            "automaticamente pelos candidatos comuns observados."
        )

    concluida("Fonte GR_CTGR_TRAN")

    # --------------------------------------------------------
    etapa("Categoria original → vigente")
    # --------------------------------------------------------
    agg_matriz = [
        F.count("*").alias("movimentos")
    ]

    if COL_VALOR:
        agg_matriz.append(
            F.sum(
                F.col(COL_VALOR)
            ).alias("valor")
        )
    else:
        agg_matriz.append(
            F.lit(None)
            .cast("double")
            .alias("valor")
        )

    matriz = (
        mov.groupBy(
            "CD_CTGR_TRAN_OGNL",
            "CD_CTGR_TRAN",
        )
        .agg(*agg_matriz)
        .orderBy(F.desc("movimentos"))
    )

    iguais = (
        mov.where(
            F.col("CD_CTGR_TRAN_OGNL")
            .eqNullSafe(
                F.col("CD_CTGR_TRAN")
            )
        )
        .count()
    )

    mudaram = QT_MOV - iguais

    RESULTADO_ESTUDO["categorias"]["original_vigente"] = (
        rows_dict(matriz, 500)
    )
    RESULTADO_ESTUDO["categorias"]["iguais"] = (
        iguais
    )
    RESULTADO_ESTUDO["categorias"]["mudaram"] = (
        mudaram
    )
    RESULTADO_ESTUDO["resumo"]["mudancas_categoria"] = (
        mudaram
    )

    # Instituições e contas onde as mudanças aparecem.
    mov_mudou = mov.where(
        ~F.col("CD_CTGR_TRAN_OGNL")
        .eqNullSafe(F.col("CD_CTGR_TRAN"))
    )

    detalhes_mudanca = {}

    if "CD_INST_PCT" in mov_mudou.columns:
        detalhes_mudanca["instituicoes"] = rows_dict(
            mov_mudou.groupBy(
                "CD_CTGR_TRAN_OGNL",
                "CD_CTGR_TRAN",
                "CD_INST_PCT",
            )
            .agg(F.count("*").alias("movimentos"))
            .orderBy(F.desc("movimentos")),
            200,
        )

    if "CONTA_PROTEGIDA" in mov_mudou.columns:
        detalhes_mudanca["contas"] = rows_dict(
            mov_mudou.groupBy(
                "CD_CTGR_TRAN_OGNL",
                "CD_CTGR_TRAN",
                "CONTA_PROTEGIDA",
            )
            .agg(F.count("*").alias("movimentos"))
            .orderBy(F.desc("movimentos")),
            200,
        )

    RESULTADO_ESTUDO["categorias"]["detalhes_mudanca"] = (
        detalhes_mudanca
    )

    concluida("Categoria original → vigente")

    # --------------------------------------------------------
    etapa("Auditoria Água")
    # --------------------------------------------------------
    if COL_DESC_CAT:
        agua_cat = (
            categorias.where(
                texto_normalizado(
                    COL_DESC_CAT
                ).contains("AGUA")
            )
            .persist(
                StorageLevel.MEMORY_AND_DISK
            )
        )

        codigos_agua = (
            agua_cat.select("CD_CTGR_TRAN")
            .where(
                F.col("CD_CTGR_TRAN").isNotNull()
            )
            .dropDuplicates()
        )

        mov_agua = (
            mov.join(
                F.broadcast(codigos_agua),
                ["CD_CTGR_TRAN"],
                "inner",
            )
            .persist(
                StorageLevel.MEMORY_AND_DISK
            )
        )

        QT_AGUA = mov_agua.count()

        desc_map = (
            categorias.groupBy("CD_CTGR_TRAN")
            .agg(
                F.concat_ws(
                    " | ",
                    F.sort_array(
                        F.collect_set(
                            F.col(
                                COL_DESC_CAT
                            ).cast("string")
                        )
                    ),
                ).alias("DESC_CAT"),
            )
        )

        agua = mov_agua.join(
            F.broadcast(
                desc_map.withColumnRenamed(
                    "DESC_CAT",
                    "DESC_CATEGORIA_VIGENTE",
                )
            ),
            ["CD_CTGR_TRAN"],
            "left",
        )

        agua = agua.join(
            F.broadcast(
                desc_map
                .withColumnRenamed(
                    "CD_CTGR_TRAN",
                    "CD_CTGR_TRAN_OGNL",
                )
                .withColumnRenamed(
                    "DESC_CAT",
                    "DESC_CATEGORIA_ORIGINAL",
                )
            ),
            ["CD_CTGR_TRAN_OGNL"],
            "left",
        )

        # Identificador local somente para auditoria visual.
        ordem = []

        for c in [
            COL_DATA,
            "NR_TRAN_INST_PCT",
            "NR_PTC",
            COL_VALOR,
        ]:
            if c and c in agua.columns:
                ordem.append(
                    F.col(c).asc_nulls_last()
                )

        if not ordem:
            ordem = [
                F.monotonically_increasing_id()
            ]

        agua = agua.withColumn(
            "ID_MOV_LOCAL",
            F.concat(
                F.lit("MOV_"),
                F.lpad(
                    F.row_number()
                    .over(
                        Window.orderBy(*ordem)
                    )
                    .cast("string"),
                    5,
                    "0",
                ),
            ),
        )

        # Campos base permitidos para auditoria.
        audit_cols = ["ID_MOV_LOCAL"]

        for c in [
            COL_DATA,
            COL_VALOR,
            "NR_TRAN_INST_PCT",
            "NR_PTC",
            "CD_INST_PCT",
            "NR_MCA_PCT_OPB",
            "CD_CTGR_TRAN_OGNL",
            "CD_CTGR_TRAN",
            "CONTA_PROTEGIDA",
            "DESC_CATEGORIA_ORIGINAL",
            "DESC_CATEGORIA_VIGENTE",
        ]:
            if (
                c
                and c in agua.columns
                and c not in audit_cols
                and (
                    coluna_relatorio_segura(c)
                    or c == "CONTA_PROTEGIDA"
                )
            ):
                audit_cols.append(c)

        # Acrescenta data/hora/natureza/estado/tipo/descrições disponíveis.
        for f in agua.schema.fields:
            c = f.name
            if (
                c in audit_cols
                or not coluna_relatorio_segura(c)
            ):
                continue

            n = c.upper()

            if isinstance(
                f.dataType,
                (DateType, TimestampType),
            ):
                audit_cols.append(c)

            elif isinstance(
                f.dataType,
                StringType,
            ) and any(
                x in n
                for x in [
                    "DS_",
                    "DESC",
                    "NAT",
                    "EST",
                    "SIT",
                    "TIP",
                    "PROD",
                    "HORA",
                    "HR_",
                ]
            ):
                audit_cols.append(c)

        # Seleciona automaticamente a hipótese de complemento
        # com melhor comportamento observado.
        rel_cmpt_validos = [
            r for r in rel_cmpt
            if r.get("status")
            == "FATO OBSERVADO"
        ]

        if rel_cmpt_validos:
            rel_cmpt_validos.sort(
                key=score_relacionamento,
                reverse=True,
            )
            melhor_rel_cmpt = (
                rel_cmpt_validos[0]
            )
            chave_cmpt_agua = (
                melhor_rel_cmpt["chaves"]
            )
        else:
            melhor_rel_cmpt = None
            chave_cmpt_agua = None

        agua_audit = agua
        cmpt_cols_audit = []

        if (
            cmpt is not None
            and chave_cmpt_agua
            and all(
                c in cmpt.columns
                and c in agua.columns
                for c in chave_cmpt_agua
            )
        ):
            exclusivos = [
                c for c in cmpt.columns
                if c not in chave_cmpt_agua
                and c not in agua.columns
                and coluna_relatorio_segura(c)
            ]

            tipos_cmpt = schema_type_map(cmpt)
            selecionados = []

            for c in exclusivos:
                n = c.upper()
                dt = tipos_cmpt.get(c)

                if isinstance(
                    dt,
                    (DateType, TimestampType),
                ) or any(
                    x in n
                    for x in [
                        "DS_",
                        "DESC",
                        "INST",
                        "NAT",
                        "TIP",
                        "CTPT",
                        "CNTP",
                    ]
                ):
                    selecionados.append(c)

            selecionados = selecionados[:12]

            proj = [
                F.col(c)
                for c in chave_cmpt_agua
            ]

            for c in selecionados:
                alias = "CMPT__" + c
                proj.append(
                    F.col(c).alias(alias)
                )
                cmpt_cols_audit.append(
                    alias
                )

            agua_audit = agua.join(
                cmpt.select(*proj),
                chave_cmpt_agua,
                "left",
            )

        # Água -> Água x outra categoria -> Água.
        agua_agua = (
            mov_agua.join(
                F.broadcast(
                    codigos_agua.select(
                        F.col(
                            "CD_CTGR_TRAN"
                        ).alias(
                            "CD_CTGR_TRAN_OGNL"
                        )
                    )
                ),
                ["CD_CTGR_TRAN_OGNL"],
                "inner",
            )
            .count()
        )

        # Consolidação categoria original -> Água.
        original_para_agua = (
            mov_agua.groupBy(
                "CD_CTGR_TRAN_OGNL"
            )
            .agg(*agg_matriz)
            .orderBy(
                F.desc("movimentos")
            )
        )

        id_tests_agua = [
            testar_chave(
                mov_agua,
                ["NR_TRAN_INST_PCT"],
                QT_AGUA,
                "NR_TRAN_INST_PCT",
            ),
            testar_chave(
                mov_agua,
                [
                    "NR_TRAN_INST_PCT",
                    "NR_PTC",
                ],
                QT_AGUA,
                "NR_TRAN_INST_PCT + NR_PTC",
            ),
        ]

        # Combinação data/hora/valor/conta.
        combo = []

        for f in mov_agua.schema.fields:
            if isinstance(
                f.dataType,
                (DateType, TimestampType),
            ):
                combo.append(f.name)
            elif (
                isinstance(f.dataType, StringType)
                and any(
                    x in f.name.upper()
                    for x in ["HORA", "HR_"]
                )
                and coluna_relatorio_segura(
                    f.name
                )
            ):
                combo.append(f.name)

        if (
            COL_VALOR
            and COL_VALOR in mov_agua.columns
        ):
            combo.append(COL_VALOR)

        if (
            "CONTA_PROTEGIDA"
            in mov_agua.columns
        ):
            combo.append(
                "CONTA_PROTEGIDA"
            )

        combo = list(
            dict.fromkeys(combo)
        )

        dup_combo = duplicidade_exata(
            mov_agua,
            combo,
        )

        if dup_combo["grupos"]:
            adicionar_anomalia(
                "FATO OBSERVADO",
                f"Água: {dup_combo['grupos']} combinação(ões) "
                "de data/hora/valor/conta aparece(m) repetida(s).",
            )

        desc_recorrentes = []

        for c in colunas_textuais_seguras(
            mov_agua
        ):
            if not any(
                x in c.upper()
                for x in ["DS_", "DESC"]
            ):
                continue

            freq = (
                mov_agua.groupBy(c)
                .agg(
                    F.count("*").alias(
                        "quantidade"
                    )
                )
                .orderBy(
                    F.desc("quantidade")
                )
            )

            top = rows_dict(
                freq,
                10,
            )

            if (
                top
                and int(
                    top[0].get(
                        "quantidade",
                        0,
                    )
                )
                > 1
            ):
                desc_recorrentes.append(
                    {
                        "coluna": c,
                        "valores": top,
                    }
                )

        cols_saida = []

        for c in (
            audit_cols
            + cmpt_cols_audit
        ):
            if (
                c in agua_audit.columns
                and c not in cols_saida
            ):
                cols_saida.append(c)

        RESULTADO_ESTUDO["agua"] = {
            "total": QT_AGUA,
            "agua_para_agua": agua_agua,
            "outra_para_agua": QT_AGUA
            - agua_agua,
            "categorias_originais": rows_dict(
                original_para_agua,
                100,
            ),
            "instituicoes": (
                rows_dict(
                    mov_agua.groupBy(
                        "CD_INST_PCT"
                    )
                    .agg(
                        F.count("*").alias(
                            "quantidade"
                        )
                    )
                    .orderBy(
                        F.desc("quantidade")
                    ),
                    100,
                )
                if "CD_INST_PCT"
                in mov_agua.columns
                else []
            ),
            "contas": (
                rows_dict(
                    mov_agua.groupBy(
                        "CONTA_PROTEGIDA"
                    )
                    .agg(
                        F.count("*").alias(
                            "quantidade"
                        )
                    )
                    .orderBy(
                        F.desc("quantidade")
                    ),
                    100,
                )
                if "CONTA_PROTEGIDA"
                in mov_agua.columns
                else []
            ),
            "identificadores": id_tests_agua,
            "combinacoes_exatamente_repetidas": dup_combo,
            "descricoes_recorrentes": desc_recorrentes,
            "chave_complemento_auditoria": chave_cmpt_agua,
            "registros": rows_dict(
                agua_audit.select(
                    *cols_saida
                ).orderBy(
                    "ID_MOV_LOCAL"
                ),
                200,
            ),
            "registros_total": QT_AGUA,
        }

        RESULTADO_ESTUDO["resumo"]["movimentacoes_agua"] = (
            QT_AGUA
        )

    else:
        RESULTADO_ESTUDO["agua"] = {
            "status": "NÃO DETERMINADO",
            "motivo": (
                "A coluna de descrição da categoria Água "
                "não pôde ser determinada automaticamente."
            ),
        }
        RESULTADO_ESTUDO["resumo"]["movimentacoes_agua"] = (
            None
        )

    concluida("Auditoria Água")

    # --------------------------------------------------------
    etapa("Resumo executivo e questões abertas")
    # --------------------------------------------------------
    RESULTADO_ESTUDO["resumo"]["tabelas_estudadas"] = (
        len(
            RESULTADO_ESTUDO[
                "fontes"
            ]
        )
    )

    RESULTADO_ESTUDO["resumo"]["hipoteses_chave_testadas"] = (
        sum(
            len(v.get("chaves", []))
            for v in RESULTADO_ESTUDO[
                "fontes"
            ].values()
            if isinstance(v, dict)
        )
    )

    rel_validos = [
        r
        for r in RESULTADO_ESTUDO[
            "relacionamentos"
        ]
        if r.get("status")
        == "FATO OBSERVADO"
    ]

    RESULTADO_ESTUDO["resumo"]["relacionamentos_testados"] = (
        len(rel_validos)
    )

    RESULTADO_ESTUDO["resumo"]["relacionamentos_com_multiplicidade"] = (
        sum(
            1
            for r in rel_validos
            if r.get("um_n", 0)
            or r.get("n_um", 0)
            or r.get("n_n", 0)
        )
    )

    adicionar_questao(
        "Qual combinação identifica definitivamente uma movimentação "
        "como chave física? O comportamento observado não substitui "
        "metadado declarado."
    )
    adicionar_questao(
        "Qual combinação identifica definitivamente uma conta "
        "como chave física? O comportamento observado não substitui "
        "metadado declarado."
    )
    adicionar_questao(
        "PK/FK/UK e restrições físicas permanecem "
        "METADADO_NAO_VALIDADO até existir objeto SYSIBM "
        "comprovadamente disponível no ambiente."
    )

    RESULTADO_ESTUDO["resumo"]["questoes_abertas"] = (
        len(
            RESULTADO_ESTUDO[
                "questoes_abertas"
            ]
        )
    )

    RESULTADO_ESTUDO["execucao_ok"] = True
    RESULTADO_ESTUDO["etapa_interrompida"] = None

    concluida("Resumo executivo e questões abertas")

    print("[ESTUDO] processamento remoto concluído.")

except Exception as exc:
    RESULTADO_ESTUDO["execucao_ok"] = False
    RESULTADO_ESTUDO["erro"] = (
        f"{type(exc).__name__}: {str(exc)}"
    )[:1200]

    try:
        registrar_erro_rotina(
            RESULTADO_ESTUDO[
                "etapa_interrompida"
            ]
            or "ESTUDO_TABELAS",
            exc,
        )
    except Exception:
        pass

    print(
        "[ESTUDO] execução incompleta; "
        "estado parcial preservado para o Markdown."
    )

In [ ]:
# ============================================================
# 6. Relatório único Markdown, sanitizado e curado
# ============================================================
def safe_text(v):
    if v is None:
        return ""

    s = str(v)

    if CLIENTE_PROTEGIDO:
        s = s.replace(
            str(CLIENTE_PROTEGIDO),
            "<PROTEGIDO>",
        )

    s = re.sub(
        r"(?i)\b(password|senha|secret|token|keytab)\s*([=:])\s*([^\s,;]+)",
        r"\1\2***",
        s,
    )

    return s

def fmt(v):
    if v is None or v == "":
        return "NÃO DETERMINADO"

    if isinstance(v, float):
        return (
            f"{v:,.2f}"
            .replace(",", "X")
            .replace(".", ",")
            .replace("X", ".")
        )

    return str(v)

def md_table(rows, cols=None, limite=50):
    if not rows:
        return "_Nenhum detalhe necessário._"

    rows = rows[:limite]

    if cols is None:
        cols = list(rows[0].keys())

    cab = "| " + " | ".join(
        safe_text(c)
        for c in cols
    ) + " |"

    sep = "| " + " | ".join(
        "---"
        for _ in cols
    ) + " |"

    corpo = []

    for row in rows:
        vals = []

        for c in cols:
            v = row.get(c, "")

            if isinstance(v, float):
                v = fmt(v)

            vals.append(
                safe_text(v)
                .replace("|", "\\|")
                .replace("\n", " ")
            )

        corpo.append(
            "| "
            + " | ".join(vals)
            + " |"
        )

    return "\n".join(
        [cab, sep] + corpo
    )

def preenchimento_md(p):
    if not p:
        return "_NÃO DETERMINADO._"

    vazias = p.get(
        "colunas_100_vazias",
        [],
    )
    parciais = p.get(
        "preenchimento_parcial",
        [],
    )

    linhas = [
        (
            "- Colunas analisadas: "
            f"**{p.get('colunas_analisadas', 0)}**."
        ),
        (
            "- Colunas 100% vazias: "
            f"**{len(vazias)}**."
        ),
    ]

    if vazias:
        linhas.append(
            "- 100% vazias: "
            + ", ".join(
                f"`{x}`"
                for x in vazias[:20]
            )
            + (
                " ..."
                if len(vazias) > 20
                else ""
            )
        )

    if parciais:
        linhas.append(
            "- Maiores percentuais de nulos:"
        )

        for x in parciais[:10]:
            linhas.append(
                f"  - `{x['coluna']}`: "
                f"{x['pct_nulos']}%."
            )

    return "\n".join(linhas)

def dominios_md(dominios):
    linhas = []

    for d in dominios or []:
        c = d.get("coluna")
        card = d.get("cardinalidade")

        if card == ">100":
            linhas.append(
                f"- `{c}`: **CARDINALIDADE > 100** — não expandida."
            )
            continue

        linhas.append(
            f"- `{c}`: **{card} valor(es)**."
        )

        vals = d.get(
            "valores",
            [],
        )

        if (
            isinstance(card, int)
            and card <= 30
            and vals
        ):
            linhas += [
                "",
                md_table(
                    vals,
                    limite=30,
                ),
                "",
            ]

        elif vals:
            top = []

            for x in vals[:10]:
                top.append(
                    f"`{safe_text(x.get(c))}` "
                    f"({x.get('quantidade')})"
                )

            linhas.append(
                "  Mais frequentes: "
                + "; ".join(top)
            )

    return (
        "\n".join(linhas)
        if linhas
        else "_Nenhum domínio disponível._"
    )

def chaves_md(testes):
    linhas = []

    for t in testes or []:
        nome = t.get(
            "nome",
            "hipótese",
        )

        if (
            t.get("status")
            != "FATO OBSERVADO"
        ):
            linhas.append(
                f"- `{nome}`: **NÃO DETERMINADO** — "
                f"{safe_text(t.get('motivo', ''))}"
            )
            continue

        linhas.append(
            f"- `{nome}`: "
            f"{t.get('linhas')} registros / "
            f"{t.get('combinacoes')} combinações / "
            f"**{t.get('repeticoes')} repetições**."
        )

        if (
            t.get("repeticoes", 0)
            and t.get("exemplos")
        ):
            linhas += [
                "",
                md_table(
                    t["exemplos"],
                    limite=10,
                ),
                "",
            ]

    return "\n".join(linhas)

def relacionamentos_md(rels):
    blocos = []

    for r in rels or []:
        nome = r.get("nome", "")
        chave = " + ".join(
            r.get("chaves", [])
        )

        if (
            r.get("status")
            != "FATO OBSERVADO"
        ):
            blocos.append(
                f"### {nome} — `{chave}`\n\n"
                f"**NÃO DETERMINADO:** "
                f"{safe_text(r.get('motivo', ''))}"
            )
            continue

        texto = (
            f"### {nome} — `{chave}`\n\n"
            f"- chaves na origem: **{r.get('chaves_origem', 0)}**\n"
            f"- chaves no destino: **{r.get('chaves_destino', 0)}**\n"
            f"- linhas da origem com chave nula: **{r.get('linhas_origem_com_chave_nula', 0)}**\n"
            f"- linhas do destino com chave nula: **{r.get('linhas_destino_com_chave_nula', 0)}**\n"
            f"- sem match: **{r.get('sem_match', 0)}**\n"
            f"- 1:1: **{r.get('um_um', 0)}**\n"
            f"- 1:N: **{r.get('um_n', 0)}**\n"
            f"- N:1: **{r.get('n_um', 0)}**\n"
            f"- N:N: **{r.get('n_n', 0)}**"
        )

        if r.get("exemplos"):
            texto += (
                "\n\nExceções relevantes:\n\n"
                + md_table(
                    r["exemplos"],
                    limite=10,
                )
            )

        blocos.append(texto)

    return "\n\n".join(blocos)

try:
    RESULTADO_ESTUDO = spark.get_from_spark(
        "RESULTADO_ESTUDO"
    )
except Exception as exc:
    gravar_execucao_incompleta(
        "Recuperação do resultado Spark",
        erro_local_sanitizado(exc),
    )
    raise

try:
    r = RESULTADO_ESTUDO
    res = r.get("resumo", {})
    fontes = r.get("fontes", {})
    linhas = []

    # 1. Identificação
    linhas += [
        "# Estudo técnico das tabelas",
        "",
        "## 1. Identificação do estudo",
        "",
        (
            f"- **Período:** "
            f"{safe_text(r['identificacao'].get('periodo'))}"
        ),
        (
            f"- **Moeda:** "
            f"{safe_text(r['identificacao'].get('moeda'))}"
        ),
        (
            f"- **Tecnologia:** "
            f"{safe_text(r['identificacao'].get('tecnologia'))}"
        ),
        (
            f"- **Situação:** "
            f"{'CONCLUÍDA' if r.get('execucao_ok') else 'INCOMPLETA'}"
        ),
        (
            "- **Tabelas:** "
            + ", ".join(
                f"`{x}`"
                for x in r[
                    "identificacao"
                ].get(
                    "tabelas",
                    [],
                )
            )
        ),
        (
            f"- **Metadado físico:** "
            f"{safe_text(r['identificacao'].get('metadado_fisico'))}"
        ),
        "",
    ]

    # 2. Resumo
    linhas += [
        "## 2. Resumo executivo técnico",
        "",
        (
            f"- {res.get('tabelas_estudadas', len(fontes))} "
            "fontes estudadas."
        ),
        (
            f"- {fmt(res.get('instituicoes_estrutura'))} "
            "instituições na estrutura de contas."
        ),
        (
            f"- {fmt(res.get('instituicoes_movimentadas'))} "
            "instituições com movimentação em julho."
        ),
        (
            f"- {fmt(res.get('movimentacoes'))} "
            "movimentações no período."
        ),
        (
            f"- {res.get('hipoteses_chave_testadas', 0)} "
            "hipóteses de chave testadas."
        ),
        (
            f"- {res.get('relacionamentos_testados', 0)} "
            "relacionamentos testados."
        ),
        (
            f"- {res.get('relacionamentos_com_multiplicidade', 0)} "
            "relacionamentos com multiplicidade."
        ),
        (
            f"- {fmt(res.get('mudancas_categoria'))} "
            "movimentações com categoria original diferente da vigente."
        ),
        (
            f"- {fmt(res.get('movimentacoes_agua'))} "
            "movimentações com categoria vigente Água."
        ),
        (
            f"- {res.get('questoes_abertas', len(r.get('questoes_abertas', [])))} "
            "questões abertas."
        ),
        "",
    ]

    # 3. Transações
    f = fontes.get(
        "TRAN_RLZD_INST_PCT",
        {},
    )

    linhas += [
        "## 3. Fonte `TRAN_RLZD_INST_PCT`",
        "",
        (
            f"**Volume observado:** "
            f"{fmt(f.get('volume'))}"
        ),
        "",
        (
            f"**Grão aparente:** "
            f"{safe_text(f.get('grao', 'NÃO DETERMINADO'))}"
        ),
        "",
        "### Mapeamento operacional observado",
        "",
        md_table(
            [f.get("mapeamento_operacional", {})]
            if f.get("mapeamento_operacional")
            else [],
            limite=1,
        ),
        "",
        "### Preenchimento",
        "",
        preenchimento_md(
            f.get("preenchimento")
        ),
        "",
        "### Hipóteses de identidade",
        "",
        chaves_md(
            f.get("chaves", [])
        ),
        "",
        "### Cardinalidade e domínios",
        "",
        dominios_md(
            f.get("dominios", [])
        ),
        "",
    ]

    # 4. Contas
    f = fontes.get(
        "INF_OPB_CT_CLI",
        {},
    )

    linhas += [
        "## 4. Fonte `INF_OPB_CT_CLI`",
        "",
        (
            f"**Volume observado:** "
            f"{fmt(f.get('volume'))}"
        ),
        "",
        (
            f"**Grão aparente:** "
            f"{safe_text(f.get('grao', 'NÃO DETERMINADO'))}"
        ),
        "",
        "### Preenchimento",
        "",
        preenchimento_md(
            f.get("preenchimento")
        ),
        "",
        "### Hipóteses de conta",
        "",
        chaves_md(
            f.get("chaves", [])
        ),
        "",
        "### Instituições, marcas e contas candidatas",
        "",
        md_table(
            f.get("instituicoes", []),
            limite=100,
        ),
        "",
        "### Cardinalidade e domínios",
        "",
        dominios_md(
            f.get("dominios", [])
        ),
        "",
    ]

    # 5. Instituições
    inst = r.get(
        "instituicoes",
        {},
    )

    linhas += [
        "## 5. Instituições do participante",
        "",
        "**FATOS OBSERVADOS:**",
        "",
        (
            "- Códigos de instituição na estrutura de contas: "
            f"**{fmt(res.get('instituicoes_estrutura'))}**."
        ),
        (
            "- Códigos de marca: "
            f"**{fmt(res.get('marcas_estrutura'))}**."
        ),
        (
            "- Instituições movimentadas em julho: "
            f"**{fmt(res.get('instituicoes_movimentadas'))}**."
        ),
        (
            "- Instituições da estrutura sem movimentação em julho: "
            f"**{fmt(inst.get('estrutura_sem_movimento'))}**."
        ),
        "",
        "### Estrutura de contas × movimentação de julho",
        "",
        md_table(
            inst.get("comparacao", []),
            limite=100,
        ),
        "",
    ]

    if inst.get("multimarcas"):
        linhas += [
            "### Instituições associadas a mais de uma marca",
            "",
            md_table(
                inst["multimarcas"],
                limite=20,
            ),
            "",
        ]

    # 6. Complementos
    f = fontes.get(
        "CMPT_TRAN_RLZD_CC",
        {},
    )

    linhas += [
        "## 6. Fonte `CMPT_TRAN_RLZD_CC`",
        "",
        (
            f"**Volume do universo dirigido:** "
            f"{fmt(f.get('volume'))}"
        ),
        "",
        (
            f"**Estratégia de leitura:** "
            f"{safe_text(f.get('estrategia_leitura', 'NÃO DETERMINADO'))}"
        ),
        "",
        (
            f"**Grão aparente:** "
            f"{safe_text(f.get('grao', 'NÃO DETERMINADO'))}"
        ),
        "",
        "### Preenchimento",
        "",
        preenchimento_md(
            f.get("preenchimento")
        ),
        "",
        "### Campos e domínios observados",
        "",
        dominios_md(
            f.get("dominios", [])
        ),
        "",
    ]

    # 7. Categorias
    fc = fontes.get(
        "CTGR_TRAN_OPB",
        {},
    )
    fg = fontes.get(
        "GR_CTGR_TRAN",
        {},
    )

    linhas += [
        "## 7. Categorias",
        "",
        (
            "- Registros de categoria necessários: "
            f"**{fmt(fc.get('volume'))}**."
        ),
        (
            "- Coluna textual usada para localizar Água: "
            f"**{safe_text(fc.get('descricao_detectada', 'NÃO DETERMINADO'))}**."
        ),
        (
            "- Registros de grupo necessários: "
            f"**{fmt(fg.get('volume'))}**."
        ),
        (
            "- Chave candidata categoria → grupo: "
            f"**{safe_text(fg.get('chave_relacionamento_candidata', 'NÃO DETERMINADO'))}**."
        ),
        "",
        "### Hipótese de chave de categoria",
        "",
        chaves_md(
            fc.get("chaves", [])
        ),
        "",
    ]

    variacoes = [
        x
        for x in fc.get(
            "variacoes_por_codigo",
            [],
        )
        if x.get(
            "codigos_com_variacao",
            0,
        )
    ]

    if variacoes:
        linhas += [
            "### Ambiguidades observadas no mesmo código",
            "",
        ]

        for x in variacoes:
            linhas.append(
                f"- `{x['coluna']}`: "
                f"{x['codigos_com_variacao']} código(s) "
                "com mais de um valor."
            )

        linhas.append("")

    # 8. Original -> vigente
    cat = r.get(
        "categorias",
        {},
    )
    ov = cat.get(
        "original_vigente",
        [],
    )

    ov_cols = (
        [
            c
            for c in [
                "CD_CTGR_TRAN_OGNL",
                "CD_CTGR_TRAN",
                "movimentos",
                "valor",
            ]
            if ov
            and c in ov[0]
        ]
        if ov
        else None
    )

    linhas += [
        "## 8. Original → vigente",
        "",
        (
            "- Permaneceram iguais: "
            f"**{fmt(cat.get('iguais'))}**."
        ),
        (
            "- Mudaram: "
            f"**{fmt(cat.get('mudaram'))}**."
        ),
        "",
        md_table(
            ov,
            cols=ov_cols,
            limite=100,
        ),
        "",
    ]

    detalhes_mudanca = cat.get(
        "detalhes_mudanca",
        {},
    )

    if detalhes_mudanca.get("instituicoes"):
        linhas += [
            "### Mudanças por instituição",
            "",
            md_table(
                detalhes_mudanca[
                    "instituicoes"
                ],
                limite=100,
            ),
            "",
        ]

    if detalhes_mudanca.get("contas"):
        linhas += [
            "### Mudanças por conta protegida",
            "",
            md_table(
                detalhes_mudanca[
                    "contas"
                ],
                limite=100,
            ),
            "",
        ]

    # 9. Água
    agua = r.get(
        "agua",
        {},
    )

    linhas += [
        "## 9. Auditoria “Água”",
        "",
    ]

    if (
        agua.get("status")
        == "NÃO DETERMINADO"
    ):
        linhas += [
            (
                f"**NÃO DETERMINADO:** "
                f"{safe_text(agua.get('motivo'))}"
            ),
            "",
        ]
    else:
        linhas += [
            (
                f"- Total: "
                f"**{agua.get('total', 0)}**."
            ),
            (
                f"- Água → Água: "
                f"**{agua.get('agua_para_agua', 0)}**."
            ),
            (
                f"- Outra categoria → Água: "
                f"**{agua.get('outra_para_agua', 0)}**."
            ),
            (
                "- Chave de complemento usada apenas para auditoria: "
                f"**{safe_text(agua.get('chave_complemento_auditoria'))}**."
            ),
            "",
            "### Categorias originais envolvidas",
            "",
            md_table(
                agua.get(
                    "categorias_originais",
                    [],
                ),
                limite=100,
            ),
            "",
            "### Instituições",
            "",
            md_table(
                agua.get(
                    "instituicoes",
                    [],
                ),
                limite=100,
            ),
            "",
            "### Contas protegidas",
            "",
            md_table(
                agua.get(
                    "contas",
                    [],
                ),
                limite=100,
            ),
            "",
            "### Identificadores físicos candidatos",
            "",
            chaves_md(
                agua.get(
                    "identificadores",
                    [],
                )
            ),
            "",
            "### Combinações idênticas de data/hora/valor/conta",
            "",
            (
                "- Grupos repetidos: "
                f"**{agua.get('combinacoes_exatamente_repetidas', {}).get('grupos', 0)}**."
            ),
            (
                "- Linhas envolvidas: "
                f"**{agua.get('combinacoes_exatamente_repetidas', {}).get('linhas', 0)}**."
            ),
            "",
        ]

        ex_dup = agua.get(
            "combinacoes_exatamente_repetidas",
            {},
        ).get(
            "exemplos",
            [],
        )

        if ex_dup:
            linhas += [
                md_table(
                    ex_dup,
                    limite=10,
                ),
                "",
            ]

        if agua.get(
            "descricoes_recorrentes"
        ):
            linhas += [
                "### Descrições recorrentes",
                "",
            ]

            for item in agua[
                "descricoes_recorrentes"
            ]:
                linhas += [
                    (
                        f"**{safe_text(item.get('coluna'))}**"
                    ),
                    "",
                    md_table(
                        item.get(
                            "valores",
                            [],
                        ),
                        limite=10,
                    ),
                    "",
                ]

        linhas += [
            "### Registros necessários para explicar o fenômeno",
            "",
            md_table(
                agua.get(
                    "registros",
                    [],
                ),
                limite=200,
            ),
            "",
        ]

        if (
            agua.get(
                "registros_total",
                0,
            )
            > 200
        ):
            linhas += [
                (
                    f"_O universo contém "
                    f"{agua['registros_total']} registros. "
                    "O relatório apresenta no máximo 200 linhas; "
                    "o universo completo permaneceu no Spark durante o estudo._"
                ),
                "",
            ]

    # 10. Relacionamentos
    linhas += [
        "## 10. Relacionamentos",
        "",
        relacionamentos_md(
            r.get(
                "relacionamentos",
                [],
            )
        ),
        "",
    ]

    # 11. Anomalias
    linhas += [
        "## 11. Anomalias e achados",
        "",
    ]

    if r.get("anomalias"):
        for a in r[
            "anomalias"
        ]:
            linhas.append(
                f"- **{safe_text(a.get('classe'))}:** "
                f"{safe_text(a.get('texto'))}"
            )
    else:
        linhas.append(
            "_Nenhuma anomalia adicional foi registrada automaticamente._"
        )

    linhas.append("")

    # 12. Questões abertas
    linhas += [
        "## 12. Questões abertas",
        "",
    ]

    if r.get(
        "questoes_abertas"
    ):
        linhas.extend(
            f"- {safe_text(x)}"
            for x in r[
                "questoes_abertas"
            ]
        )
    else:
        linhas.append(
            "_Nenhuma questão aberta foi registrada._"
        )

    # Falha parcial
    if not r.get("execucao_ok"):
        linhas += [
            "",
            "## EXECUÇÃO INCOMPLETA",
            "",
            (
                f"**Etapa interrompida:** "
                f"{safe_text(r.get('etapa_interrompida'))}"
            ),
            "",
            (
                f"**Motivo técnico sanitizado:** "
                f"{safe_text(r.get('erro'))}"
            ),
            "",
            "**Seções concluídas antes da falha:**",
        ]

        secoes = r.get(
            "secoes_concluidas",
            [],
        )

        if secoes:
            linhas.extend(
                f"- {safe_text(x)}"
                for x in secoes
            )
        else:
            linhas.append("- Nenhuma.")

    texto_final = (
        "\n".join(linhas).strip()
        + "\n"
    )

    # Gate final: o identificador do participante jamais sai no Markdown.
    if CLIENTE_PROTEGIDO:
        texto_final = texto_final.replace(
            str(CLIENTE_PROTEGIDO),
            "<PROTEGIDO>",
        )

    OUTPUT_MD.write_text(
        texto_final,
        encoding="utf-8",
    )

    print(
        f"Relatório final gravado: {OUTPUT_MD}"
    )

except Exception as exc:
    gravar_execucao_incompleta(
        "Consolidação do Markdown",
        erro_local_sanitizado(exc),
        RESULTADO_ESTUDO.get(
            "secoes_concluidas",
            [],
        ),
    )
    raise

## Resultado esperado

Ao término, revise somente:

`estudo_tabelas_resultado.md`

O notebook não grava CSV, Excel, Parquet ou tabelas auxiliares.

Se qualquer etapa falhar, o mesmo Markdown registra a etapa interrompida, motivo sanitizado e seções já concluídas.